In [ ]:
# ── 1. Install minimal deps for this project (no Kaggle core overrides) ───
import subprocess, sys, importlib

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def ensure(import_name, *pip_args):
    try:
        importlib.import_module(import_name)
        print(f'✓ {import_name} already installed')
    except Exception:
        print(f'Installing {pip_args[-1]} ...')
        pip_install(*pip_args)

# Keep Kaggle's preinstalled numpy/pandas untouched (avoid ABI/runtime breakage)
ensure('deap', 'deap')
ensure('tabulate', 'tabulate')
ensure('openpyxl', 'openpyxl')

# mealpy requirements are strict; install without dependency resolution + required helper
ensure('opfunu', '--no-deps', 'opfunu>=1.0.0')
ensure('mealpy', '--no-deps', 'mealpy==3.0.3')

import numpy as np, pandas as pd
print(f'✅ numpy={np.__version__}, pandas={pd.__version__}')
print('✅ mealpy/opfunu installed without touching Kaggle core stack.')
print('If you previously modified numpy/pandas in this session, restart runtime once before continuing.')

In [ ]:
# ── 2. Copy source files to /kaggle/working so they are importable ────────
import os, shutil, glob

SRC_DATASET = '/kaggle/input/ransomware-src'
WORK_DIR    = '/kaggle/working'

if not os.path.isdir(SRC_DATASET):
    raise FileNotFoundError(
        f'Source dataset not found at {SRC_DATASET}.\n'
        'Please add the "ransomware-src" dataset to this notebook.'
    )

for py_file in glob.glob(f'{SRC_DATASET}/*.py'):
    dst = os.path.join(WORK_DIR, os.path.basename(py_file))
    shutil.copy2(py_file, dst)
    print(f'  copied: {os.path.basename(py_file)}')

# Make working dir importable
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print('\n✅  Source files ready')

In [ ]:
# ── 3. Choose dataset (general: RBA/WPD/PEHF/RISS) ───────────────────────
# Select one:
#   '1' -> RBA   (Excel)
#   '2' -> WPD   (Excel)
#   '3' -> PEHF  (CSV)
#   '4' -> RISS  (CSV, no header)
DATASET_IDX = '4'

# Optional: set this manually if auto-detect picks the wrong file.
# Example:
# MANUAL_DATA_PATH = '/kaggle/input/my-dataset/WPD.xlsx'
MANUAL_DATA_PATH = None

import os, glob
import pandas as pd

DATASET_HINTS = {
    '1': ['RBA.xlsx'],
    '2': ['WPD.xlsx'],
    '3': ['PEHF.csv'],
    '4': ['RISS.csv'],
}

hints = DATASET_HINTS.get(DATASET_IDX)
if hints is None:
    raise ValueError(f'Unsupported DATASET_IDX={DATASET_IDX}. Use 1, 2, 3, or 4.')

if MANUAL_DATA_PATH is not None:
    DATA_PATH = MANUAL_DATA_PATH
else:
    # Auto-find selected dataset inside /kaggle/input
    matches = []
    for name in hints:
        matches.extend(glob.glob(f'/kaggle/input/**/{name}', recursive=True))

    if not matches:
        raise FileNotFoundError(
            f'Could not find dataset file for idx={DATASET_IDX}. Tried names: {hints}'
        )

    print('Candidate files found:')
    for m in matches:
        print(f'  - {m}')

    DATA_PATH = matches[0]

print(f'✅  Selected dataset idx={DATASET_IDX}')
print(f'✅  DATA_PATH = {DATA_PATH}')

# Quick sanity check preview
if DATASET_IDX == '4':
    df_check = pd.read_csv(DATA_PATH, header=None, nrows=3)
else:
    if DATA_PATH.lower().endswith('.xlsx'):
        df_check = pd.read_excel(DATA_PATH, nrows=3)
    else:
        df_check = pd.read_csv(DATA_PATH, nrows=3)

print(f'Preview shape (first rows): {df_check.shape}')
print(f'Columns sample: {list(df_check.columns)[:20]}')

if DATASET_IDX == '2':
    print("\nWPD check -> expecting target column: 'Benign'")
    print(f"'Benign' in columns? {'Benign' in df_check.columns}")
    if 'Benign' not in df_check.columns:
        raise KeyError(
            "The selected Kaggle WPD file does not contain a 'Benign' column. "
            "Set MANUAL_DATA_PATH to the correct WPD.xlsx file."
        )

In [ ]:
# ── 4. Suppress TF noise ──────────────────────────────────────────────────
import os, sys, warnings

# Suppress TensorFlow C++ logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Suppress TensorFlow logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# Suppress absl logging
import logging
logging.getLogger('absl').setLevel(logging.ERROR)

# Redirect stderr to suppress lower-level logs
import io
stderr_backup = sys.stderr
sys.stderr = io.StringIO()

print(f'TensorFlow {tf.__version__}')
sys.stderr = stderr_backup

In [ ]:
# ── 5. Import project modules ─────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler,MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

from load_data import load_data

print('✅  Project modules imported')


In [ ]:
# ── 6. Define MODEL class (mirrors main.py, dataset-generic) ──────────────
BASE_OUT = '/kaggle/working'

class MODEL:
    def __init__(self, data_path):
        self.saved_model_path = os.path.join(BASE_OUT, 'models', 'saved', 'PremierLeague.keras')
        self.saved_data_path  = os.path.join(BASE_OUT, 'data',   'processed', 'PremierLeague_processed_data.csv')
        self.data_path  = data_path
        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None
        self.n_features = self.n_classes = None
        self.model   = None
        self.smootheringScaler = MinMaxScaler()
        self.scaler  = StandardScaler()
        self.label_encoder  = LabelEncoder()
        self.result_encoder = LabelEncoder()
        self.home_encoder   = LabelEncoder()
        self.away_encoder   = LabelEncoder()
        # GA parameters
        self.population_size = 6
        self.generations     = 10
        self.crossover_prob  = 0.85
        self.mutation_prob   = 0.15
        # Paramètres GWO optimisés pour portabilité
        self.target_evaluations = 20
        self.pop_size = 5
        # Results
        self.best_individual = None
        self.best_fitness    = 0
        self.best_metrics    = {}
        self.logbook         = None

obj = MODEL(DATA_PATH)
print('✅  MODEL object created')
print(f'   DATASET_IDX: {DATASET_IDX}')
print(f'   data_path  : {obj.data_path}')
print(f'   GA settings: population={obj.population_size}, generations={obj.generations}')

In [ ]:
DATASET_TAG = {'1': 'RBA', '2': 'WPD', '3': 'PEHF', '4': 'RISS'}.get(DATASET_IDX, f'idx{DATASET_IDX}')

In [ ]:
# ── 7. Load & preprocess data (based on DATASET_IDX) ──────────────────────
# For Excel datasets on Kaggle, normalize headers first so static target names
# like 'Family' and 'Benign' match exactly.
if DATASET_IDX in ('1', '2') and str(obj.data_path).lower().endswith('.xlsx'):
    _excel_df = pd.read_excel(obj.data_path, engine='openpyxl')
    _excel_df.columns = [str(c).replace('\ufeff', '').strip() for c in _excel_df.columns]
    _normalized_path = os.path.join(BASE_OUT, f'normalized_{DATASET_TAG.lower()}.xlsx')
    _excel_df.to_excel(_normalized_path, index=False, engine='openpyxl')
    obj.data_path = _normalized_path
    print(f'✅  Normalized Excel headers and saved temporary copy to: {obj.data_path}')
    print(f'   Columns sample: {list(_excel_df.columns)[:20]}')

load_data(obj, idx=DATASET_IDX)

print(f'\nSplit summary:')
print(f'  X_train : {obj.X_train.shape}  |  classes in y_train : {np.unique(obj.y_train)}')
print(f'  X_val   : {obj.X_val.shape}')
print(f'  X_test  : {obj.X_test.shape}')
print(f'  n_features : {obj.n_features}')
print(f'  n_classes  : {obj.n_classes}')

In [ ]:
# ── 8. Initialise GA run — memory-safe settings for Kaggle ────────────────
import copy, os, pickle, gc
from time import time
import tensorflow as tf
from GA             import run_ga_optimization
from eval           import evaluate_best_model
from print_resault  import display_results
from compare_models import rank_models, print_final_ranking, generate_charts, save_results_log

# Keep outputs separated per dataset
CHART_DIR = os.path.join(BASE_OUT, f'ga_{DATASET_TAG.lower()}')
os.makedirs(CHART_DIR, exist_ok=True)
PARTIAL_RESULTS_PATH = os.path.join(CHART_DIR, f'ga_partial_results_{DATASET_TAG.lower()}.pkl')

# Save the flat (2-D) arrays once so each model cell can restore them
_X_train_2d = obj.X_train.copy()
_X_val_2d   = obj.X_val.copy()
_X_test_2d  = obj.X_test.copy()

print(f"GA settings from obj: population={obj.population_size}, generations={obj.generations}, crossover={obj.crossover_prob}, mutation={obj.mutation_prob}")

def _save_partial_results():
    with open(PARTIAL_RESULTS_PATH, 'wb') as f:
        pickle.dump(all_results, f)
    print(f'💾  Partial results saved → {PARTIAL_RESULTS_PATH}')

def _load_partial_results():
    if os.path.exists(PARTIAL_RESULTS_PATH):
        with open(PARTIAL_RESULTS_PATH, 'rb') as f:
            return pickle.load(f)
    return {}

# Shared dict — each model cell below adds its entry here
all_results = _load_partial_results()
if all_results:
    print(f'📦  Loaded {len(all_results)} previously finished model(s) for {DATASET_TAG}')
else:
    print(f'📦  No previous partial results found for {DATASET_TAG}')

def _run_one_model(obj, model_type):
    """Run GA + evaluate for a single model, store result in all_results."""
    print(f'\n{"═"*70}')
    print(f'  🚀  GA — {model_type}  |  Dataset: {DATASET_TAG}')
    print('═'*70)

    # Cleanup before each run
    gc.collect()
    tf.keras.backend.clear_session()

    # Restore flat arrays (RNN/LSTM reshape internally; prevent cross-contamination)
    obj.X_train = _X_train_2d.copy()
    obj.X_val   = _X_val_2d.copy()
    obj.X_test  = _X_test_2d.copy()

    # Reset best-* fields
    obj.best_individual = None
    obj.best_fitness    = 0.0
    obj.best_metrics    = {}

    print(f"Using settings: population={obj.population_size}, generations={obj.generations}, crossover={obj.crossover_prob}, mutation={obj.mutation_prob}")

    try:
        t0 = time()
        exec_time = run_ga_optimization(obj, test=model_type)

        evaluate_best_model(obj, test=model_type)
        display_results(obj, exec_time, test=model_type, method='GA')

        all_results[model_type] = {
            'best_individual': copy.deepcopy(obj.best_individual),
            'best_fitness'   : obj.best_fitness,
            'best_metrics'   : copy.deepcopy(obj.best_metrics),
            'execution_time' : exec_time,
            'logbook'        : copy.deepcopy(getattr(obj, 'logbook', None)),
            'error'          : None,
        }
        _save_partial_results()
        print(f'✅  {model_type} done  |  F1={obj.best_metrics.get("f1_score", 0):.4f}')

    except Exception as exc:
        print(f'❌  {model_type} failed: {exc}')
        all_results[model_type] = {
            'best_individual': None,
            'best_fitness'   : 0.0,
            'best_metrics'   : {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0},
            'execution_time' : 0.0,
            'logbook'        : None,
            'error'          : str(exc),
        }
        _save_partial_results()
    finally:
        # Cleanup after each run
        gc.collect()
        tf.keras.backend.clear_session()

print('✅ Init done — run each model cell below individually')
print(f'   Dataset: {DATASET_TAG} | outputs folder: {CHART_DIR}')
print(f'   Shared GA settings: population={obj.population_size}, generations={obj.generations}, crossover={obj.crossover_prob}, mutation={obj.mutation_prob}')

In [ ]:
# ── 9. (Recommended) Unified GA AutoML run on Kaggle ──────────────────────
# This runs one GA that searches model type + hyperparameters,
# ranks with validation F1-score, and keeps unique best per model type.

from GA import run_ga_optimization
from eval import evaluate_best_model
from print_resault import display_results

# Kaggle-friendly GA settings (adjust if you want faster/slower runs)
obj.population_size = 6
obj.generations = 5
obj.crossover_prob = 0.85
obj.mutation_prob = 0.15

print('🚀 Starting GA AutoML...')
execution_time = run_ga_optimization(obj, test='AUTOML')

# Evaluate best selected model on test set + pretty final report
evaluate_best_model(obj, test='AUTOML')
display_results(obj, execution_time, test='AUTOML', method='GA')

print('✅ GA AutoML run finished')

In [ ]:
# ── 10. MLP — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'MLP')


In [ ]:
# ── 11. CNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'CNN')


In [ ]:
# ── 12. RNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'RNN')


In [ ]:
# ── 13. DNN — GA optimisation ─────────────────────────────────────────────
_run_one_model(obj, 'DNN')


In [ ]:
# ── 14. LSTM — GA optimisation ────────────────────────────────────────────
_run_one_model(obj, 'LSTM')


In [ ]:
# ── 14. Build ranking from completed runs ─────────────────────────────────
all_results = _load_partial_results()

if not all_results:
    raise ValueError('No model results found yet. Run at least one model cell first.')

ranking = rank_models(all_results)
print('✅  Ranking built')
print(f'   Models included: {list(all_results.keys())}')
print(f'   Total completed : {len(all_results)}')


In [ ]:
# ── 15. Print final comparison report ─────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

print_final_ranking(ranking)


In [ ]:
# ── 16. Generate charts at the end ────────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

generate_charts(all_results, ranking, CHART_DIR)
print(f'✅  Charts generated in {CHART_DIR}')


In [ ]:
# ── 17. Save the comparison log file ──────────────────────────────────────
if 'ranking' not in globals():
    all_results = _load_partial_results()
    ranking = rank_models(all_results)

log_path = save_results_log(all_results, ranking, CHART_DIR)
print(f'✅  Log saved → {log_path}')


In [ ]:
# ── 18. Display saved charts inline ───────────────────────────────────────
import glob
from IPython.display import display, Image

chart_paths = sorted(glob.glob(os.path.join(CHART_DIR, 'chart_*.png')))
print(f'Found {len(chart_paths)} chart(s)')
for path in chart_paths:
    print(f'  {os.path.basename(path)}')
    display(Image(filename=path))
